# DP/SFA state-space nodal (SSN) for three-phase circuits

This notebook is the three-phase counterpart of `DP_generalizedSSN_RLC`. It
validates the `dp.ph3` SSN components on the same circuits used to develop
them:

- a voltage-driven series RLC one-port, comparing the classical `dp.ph3.R/L/C`
  network against the dedicated `dp.ph3.Full_Serial_RLC`,
- a current-driven network with a shunt inductor and a series capacitor,
  comparing classical components against the generic `GenericTwoTerminalVTypeSSN`
  and `GenericTwoTerminalITypeSSN`, parametrized directly by `(A, B, C, D)`,
- the same current-driven network under a three-phase-to-ground fault, to
  exercise the SSN discontinuity handling under a switching event.

All three-phase parameters here are diagonal (no phase coupling), so each
phase reproduces the single-phase `DP_generalizedSSN_RLC` result; the point of
this notebook is to confirm that holds for the `dp.ph3` component layer, not
to repeat that derivation. The DP-SSN envelope is also reconstructed and
cross-checked against an EMT-SSN run of the same circuit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dpsimpy
from villas.dataprocessing.readtools import read_timeseries_csv

In [ ]:
param = np.eye(3)
identity3 = np.eye(3)

f0 = 50.0
vref = dpsimpy.Math.single_phase_variable_to_three_phase(complex(1.0, 0.0))

time_step = 1e-4
final_time = 0.1

log_dir = "logs"

In [ ]:
def run(sim_name, system, domain, comp, attr):
    dpsimpy.Logger.set_log_dir(f"{log_dir}/{sim_name}")
    logger = dpsimpy.Logger(sim_name)
    logger.log_attribute("y", attr, comp)

    sim = dpsimpy.Simulation(sim_name, dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_domain(domain)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.add_logger(logger)
    sim.run()

    return read_timeseries_csv(
        f"{log_dir}/{sim_name}/{sim_name}.csv", print_status=False
    )


def dp_node(name):
    return dpsimpy.dp.SimNode(name, dpsimpy.PhaseType.ABC)


def emt_node(name):
    return dpsimpy.emt.SimNode(name, dpsimpy.PhaseType.ABC)

## Series RLC across DP, EMT, and SSN

A 1 Ohm / 0.05 H / 0.01 F series RLC one-port, driven by a balanced 1 V
source. `build_rlc` assembles the classical three-phase network; `build_ssn`
replaces it with the single `Full_Serial_RLC` component, mirroring
`DP_Ph3_RLC1VS1_RC_vs_SSN.cpp`.

In [ ]:
def build_rlc(ph3, node, gnd):
    n1, n2, n3 = node("n1"), node("n2"), node("n3")
    vs = ph3.VoltageSource("vs")
    vs.set_parameters(vref)
    res = ph3.Resistor("r")
    res.set_parameters(1.0 * param)
    ind = ph3.Inductor("l")
    ind.set_parameters(0.05 * param)
    cap = ph3.Capacitor("c")
    cap.set_parameters(0.01 * param)
    vs.connect([gnd, n1])
    res.connect([n1, n2])
    ind.connect([n2, n3])
    cap.connect([n3, gnd])
    return dpsimpy.SystemTopology(f0, [n1, n2, n3], [vs, res, ind, cap]), ind


def build_ssn(ph3, node, gnd):
    n1 = node("n1")
    vs = ph3.VoltageSource("vs")
    vs.set_parameters(vref)
    rlc = ph3.Full_Serial_RLC("rlc")
    rlc.set_parameters(1.0 * param, 0.05 * param, 0.01 * param)
    vs.connect([gnd, n1])
    rlc.connect([n1, gnd])
    return dpsimpy.SystemTopology(f0, [n1], [vs, rlc]), rlc

In [ ]:
dp_gnd = dpsimpy.dp.SimNode.gnd
emt_gnd = dpsimpy.emt.SimNode.gnd

sys, comp = build_rlc(dpsimpy.dp.ph3, dp_node, dp_gnd)
dp_rlc = run("DP_Ph3_RLC_classical", sys, dpsimpy.Domain.DP, comp, "i_intf")

sys, comp = build_ssn(dpsimpy.dp.ph3, dp_node, dp_gnd)
dp_ssn = run("DP_Ph3_RLC_SSN", sys, dpsimpy.Domain.DP, comp, "i_intf")

sys, comp = build_rlc(dpsimpy.emt.ph3, emt_node, emt_gnd)
emt_rlc = run("EMT_Ph3_RLC_classical", sys, dpsimpy.Domain.EMT, comp, "i_intf")

sys, comp = build_ssn(dpsimpy.emt.ph3, emt_node, emt_gnd)
emt_ssn = run("EMT_Ph3_RLC_SSN", sys, dpsimpy.Domain.EMT, comp, "i_intf")

`EMT::Ph3::VoltageSource` scales its reference by `RMS3PH_TO_PEAK1PH`
(`sqrt(2/3)`) to convert an RMS three-phase reference to a per-phase peak
instantaneous amplitude; `DP::Ph3::VoltageSource` uses the reference directly
as the complex envelope amplitude. The EMT waveform is divided by that factor
before comparing it to the DP envelope reconstructed with `frequency_shift`,
otherwise the mismatch looks like a component bug but is pure unit
convention.

In [ ]:
phase = "0"
dp_rlc_a = dp_rlc["y_" + phase]
dp_ssn_a = dp_ssn["y_" + phase]
emt_rlc_a = emt_rlc["y_" + phase]
emt_ssn_a = emt_ssn["y_" + phase]

dp_rlc_emt = dp_rlc_a.frequency_shift(f0)
dp_ssn_emt = dp_ssn_a.frequency_shift(f0)
emt_rlc_scaled = emt_rlc_a.values / dpsimpy.RMS3PH_TO_PEAK1PH
emt_ssn_scaled = emt_ssn_a.values / dpsimpy.RMS3PH_TO_PEAK1PH

plt.figure(figsize=(9, 4))
plt.plot(
    emt_rlc_a.time, emt_rlc_scaled, color="0.6", linewidth=3, label="EMT classical"
)
plt.plot(emt_ssn_a.time, emt_ssn_scaled, linestyle="--", label="EMT-SSN")
plt.plot(
    dp_rlc_emt.time,
    dp_rlc_emt.values,
    linestyle=":",
    label="DP classical (reconstructed)",
)
plt.plot(
    dp_ssn_emt.time, dp_ssn_emt.values, linestyle="-.", label="DP-SSN (reconstructed)"
)
plt.xlabel("time [s]")
plt.ylabel("phase A inductor current [A]")
plt.title("Series RLC, phase A inductor current: classical vs SSN, EMT vs DP")
plt.legend()
plt.show()

In [ ]:
for ph in "012":
    max_env_err = np.max(np.abs(dp_ssn["y_" + ph].values - dp_rlc["y_" + ph].values))
    print(f"phase {ph}: DP-SSN vs DP classical, max |envelope error| =", max_env_err)
    assert max_env_err < 1e-9

rmse = np.sqrt(np.mean((dp_ssn_emt.values - emt_ssn_scaled) ** 2))
print("phase A: DP-SSN reconstructed vs EMT-SSN, RMSE:", rmse)
assert rmse < 1e-3

## Generic V-type and I-type SSN in a three-phase network

A current source feeds a resistive network with a shunt inductor and a series
capacitor. The classical network is compared against the same topology with
the inductor as a generic V-type SSN component (`x = i_abc`, `u = v_abc`,
`y = i_abc`) and the capacitor as a generic I-type SSN component
(`x = vC_abc`, `u = i_abc`, `y = v_abc`), mirroring
`DP_Ph3_R3C1L1CS1_RC_vs_SSN.cpp`.

In [ ]:
def build_r3c1l1(ph3, node, gnd):
    n1, n2 = node("n1"), node("n2")
    cs = ph3.CurrentSource("cs")
    cs.set_parameters(vref)
    r1 = ph3.Resistor("r1")
    r1.set_parameters(10 * param)
    r2 = ph3.Resistor("r2")
    r2.set_parameters(param)
    r3 = ph3.Resistor("r3")
    r3.set_parameters(5 * param)
    ind = ph3.Inductor("l1")
    ind.set_parameters(0.02 * param)
    cap = ph3.Capacitor("c1")
    cap.set_parameters(0.001 * param)
    cs.connect([gnd, n1])
    r1.connect([n2, n1])
    r2.connect([n2, gnd])
    r3.connect([n2, gnd])
    ind.connect([n2, gnd])
    cap.connect([n1, n2])
    return dpsimpy.SystemTopology(f0, [n1, n2], [cs, r1, r2, r3, ind, cap]), ind, cap


def build_r3c1l1_ssn(ph3, node, gnd):
    n1, n2 = node("n1"), node("n2")
    cs = ph3.CurrentSource("cs")
    cs.set_parameters(vref)
    r1 = ph3.Resistor("r1")
    r1.set_parameters(10 * param)
    r2 = ph3.Resistor("r2")
    r2.set_parameters(param)
    r3 = ph3.Resistor("r3")
    r3.set_parameters(5 * param)

    inductance = 0.02 * param
    genv = ph3.GenericTwoTerminalVTypeSSN("l1_gen")
    genv.set_parameters(
        np.zeros((3, 3)), np.linalg.inv(inductance), identity3, np.zeros((3, 3))
    )

    capacitance = 0.001 * param
    geni = ph3.GenericTwoTerminalITypeSSN("c1_gen")
    geni.set_parameters(
        np.zeros((3, 3)), np.linalg.inv(capacitance), identity3, np.zeros((3, 3))
    )

    cs.connect([gnd, n1])
    r1.connect([n2, n1])
    r2.connect([n2, gnd])
    r3.connect([n2, gnd])
    genv.connect([n2, gnd])
    geni.connect([n1, n2])
    return (
        dpsimpy.SystemTopology(f0, [n1, n2], [cs, r1, r2, r3, genv, geni]),
        genv,
        geni,
    )

In [ ]:
sys, ind, cap = build_r3c1l1(dpsimpy.dp.ph3, dp_node, dp_gnd)
dp_r3c1l1_i = run("DP_Ph3_R3C1L1_classical_I", sys, dpsimpy.Domain.DP, ind, "i_intf")

sys, ind, cap = build_r3c1l1(dpsimpy.dp.ph3, dp_node, dp_gnd)
dp_r3c1l1_v = run("DP_Ph3_R3C1L1_classical_V", sys, dpsimpy.Domain.DP, cap, "v_intf")

sys, genv, geni = build_r3c1l1_ssn(dpsimpy.dp.ph3, dp_node, dp_gnd)
dp_r3c1l1_ssn_i = run("DP_Ph3_R3C1L1_SSN_I", sys, dpsimpy.Domain.DP, genv, "i_intf")

sys, genv, geni = build_r3c1l1_ssn(dpsimpy.dp.ph3, dp_node, dp_gnd)
dp_r3c1l1_ssn_v = run("DP_Ph3_R3C1L1_SSN_V", sys, dpsimpy.Domain.DP, geni, "v_intf")

for ph in "012":
    err_i = np.max(
        np.abs(dp_r3c1l1_ssn_i["y_" + ph].values - dp_r3c1l1_i["y_" + ph].values)
    )
    err_v = np.max(
        np.abs(dp_r3c1l1_ssn_v["y_" + ph].values - dp_r3c1l1_v["y_" + ph].values)
    )
    print(
        f"phase {ph}: inductor current max |error| = {err_i:.3e}, capacitor voltage max |error| = {err_v:.3e}"
    )
    assert err_i < 1e-9
    assert err_v < 1e-9

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(
    dp_r3c1l1_i["y_0"].time,
    dp_r3c1l1_i["y_0"].values.real,
    color="0.6",
    linewidth=3,
    label="classical, Re",
)
plt.plot(
    dp_r3c1l1_ssn_i["y_0"].time,
    dp_r3c1l1_ssn_i["y_0"].values.real,
    linestyle="--",
    label="generic V-type SSN, Re",
)
plt.xlabel("time [s]")
plt.ylabel("phase A inductor current envelope [A]")
plt.title("Shunt inductor as generic V-type SSN: classical vs SSN")
plt.legend()
plt.show()

## SSN under a three-phase fault

The same current-driven network, with a three-phase-to-ground `SeriesSwitch`
fault at node n2 between 0.03 s and 0.06 s, mirrors
`DP_Ph3_R3C1L1CS1_RC_vs_SSN_Fault.cpp`. This exercises the SSN model through a
switching discontinuity rather than only a steady excitation.

In [ ]:
switch_open_r = 1e6
switch_closed_r = 0.001
fault_start = 0.03
fault_end = 0.06


def build_r3c1l1_fault(ph3, node, gnd, ssn):
    n1, n2 = node("n1"), node("n2")
    cs = ph3.CurrentSource("cs")
    cs.set_parameters(vref)
    r1 = ph3.Resistor("r1")
    r1.set_parameters(10 * param)
    r2 = ph3.Resistor("r2")
    r2.set_parameters(param)
    r3 = ph3.Resistor("r3")
    r3.set_parameters(5 * param)

    if ssn:
        inductance = 0.02 * param
        ind = ph3.GenericTwoTerminalVTypeSSN("l1_gen")
        ind.set_parameters(
            np.zeros((3, 3)), np.linalg.inv(inductance), identity3, np.zeros((3, 3))
        )
        capacitance = 0.001 * param
        cap = ph3.GenericTwoTerminalITypeSSN("c1_gen")
        cap.set_parameters(
            np.zeros((3, 3)), np.linalg.inv(capacitance), identity3, np.zeros((3, 3))
        )
    else:
        ind = ph3.Inductor("l1")
        ind.set_parameters(0.02 * param)
        cap = ph3.Capacitor("c1")
        cap.set_parameters(0.001 * param)

    fault = ph3.SeriesSwitch("fault")
    fault.set_parameters(switch_open_r, switch_closed_r, False)

    cs.connect([gnd, n1])
    r1.connect([n2, n1])
    r2.connect([n2, gnd])
    r3.connect([n2, gnd])
    ind.connect([n2, gnd])
    cap.connect([n1, n2])
    fault.connect([n2, gnd])
    return (
        dpsimpy.SystemTopology(f0, [n1, n2], [cs, r1, r2, r3, ind, cap, fault]),
        ind,
        fault,
    )


def run_fault(sim_name, system, comp, fault):
    dpsimpy.Logger.set_log_dir(f"{log_dir}/{sim_name}")
    logger = dpsimpy.Logger(sim_name)
    logger.log_attribute("y", "i_intf", comp)

    sim = dpsimpy.Simulation(sim_name, dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_domain(dpsimpy.Domain.DP)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.add_logger(logger)
    sim.do_system_matrix_recomputation(True)
    sim.add_event(dpsimpy.event.SwitchEvent(fault_start, fault, True))
    sim.add_event(dpsimpy.event.SwitchEvent(fault_end, fault, False))
    sim.run()

    return read_timeseries_csv(
        f"{log_dir}/{sim_name}/{sim_name}.csv", print_status=False
    )

In [ ]:
sys, ind, fault = build_r3c1l1_fault(dpsimpy.dp.ph3, dp_node, dp_gnd, ssn=False)
dp_fault_rc = run_fault("DP_Ph3_R3C1L1_Fault_RC", sys, ind, fault)

sys, ind, fault = build_r3c1l1_fault(dpsimpy.dp.ph3, dp_node, dp_gnd, ssn=True)
dp_fault_ssn = run_fault("DP_Ph3_R3C1L1_Fault_SSN", sys, ind, fault)

plt.figure(figsize=(9, 4))
plt.plot(
    dp_fault_rc["y_0"].time,
    dp_fault_rc["y_0"].values.real,
    color="0.6",
    linewidth=3,
    label="classical, Re",
)
plt.plot(
    dp_fault_ssn["y_0"].time,
    dp_fault_ssn["y_0"].values.real,
    linestyle="--",
    label="SSN, Re",
)
plt.axvspan(fault_start, fault_end, color="0.9", label="fault")
plt.xlabel("time [s]")
plt.ylabel("phase A inductor current envelope [A]")
plt.title(
    "Shunt inductor current envelope through a three-phase fault: classical vs SSN"
)
plt.legend()
plt.show()

In [ ]:
for ph in "012":
    err = np.max(np.abs(dp_fault_ssn["y_" + ph].values - dp_fault_rc["y_" + ph].values))
    print(f"phase {ph}: RC vs SSN through the fault, max |envelope error| =", err)
    assert err < 1e-6